In [9]:
from IPython.display import display
import pandas as pd

# ==========================================
# [1단계] 창고 제외 v3 데이터 로드 및 구조 확인 (display 적용)
# ==========================================
file_path = "data_1127_정제_학습용데이터-v3.xlsx"
xls = pd.ExcelFile(file_path)

display("시트 목록:", xls.sheet_names)

weekly_df = pd.read_excel(xls, sheet_name="주별_학습용집계")

# '합계' 요약행이 섞여 있다면 제외
weekly_df = weekly_df[
    weekly_df["기준주차_시작일"].notna()
    & (weekly_df["기준주차_시작일"].astype(str) != "합계")
].copy()
weekly_df["기준주차_시작일"] = pd.to_datetime(weekly_df["기준주차_시작일"])
weekly_df = weekly_df.sort_values(["상품명", "기준주차_시작일"]).reset_index(
    drop=True
)

info_df = pd.DataFrame([{
    "데이터_행수": len(weekly_df),
    "데이터_열수": weekly_df.shape[1],
    "시작주차": weekly_df["기준주차_시작일"].min().strftime("%Y-%m-%d"),
    "종료주차": weekly_df["기준주차_시작일"].max().strftime("%Y-%m-%d"),
    "고유_주차수": weekly_df["기준주차_시작일"].nunique(),
    "고유_상품수": weekly_df["상품명"].nunique(),
    "결측치_합계": int(weekly_df.isna().sum().sum()),
}])

display("=== [1단계] 데이터 기본 정보 ===")
display(info_df)

num_cols = [
    "입고수량",
    "주문수량",
    "판매주문수량",
    "판매수량",
    "주문_판매_갭",
    "순재고변동",
]
desc_df = (
    weekly_df[num_cols]
    .describe()
    .T[["count", "mean", "std", "min", "50%", "max"]]
    .round(2)
)

display("=== 주요 수치 컬럼 기초통계 ===")
display(desc_df)

display("=== 상위 5행 미리보기 ===")
display(weekly_df.head(5))

'시트 목록:'

['거래내역_정제본', '주별_학습용집계', '요약_및_정제노트']

'=== [1단계] 데이터 기본 정보 ==='

,데이터_행수,데이터_열수,시작주차,종료주차,고유_주차수,고유_상품수,결측치_합계
0,1764,10,2023-01-02,2026-11-30,191,278,0


'=== 주요 수치 컬럼 기초통계 ==='

,count,mean,std,min,50%,max
입고수량,1764.0,2472.92,16872.39,0.0,260.0,430000.0
주문수량,1764.0,5103.35,33683.35,1.0,630.0,860000.0
판매주문수량,1764.0,2630.42,16905.60,0.0,270.0,430000.0
판매수량,1764.0,2462.53,16894.78,0.0,250.0,430000.0
주문_판매_갭,1764.0,167.89,381.83,0.0,12.0,2690.0
순재고변동,1764.0,10.39,2460.97,-40000.0,0.0,40000.0


'=== 상위 5행 미리보기 ==='

,기준주차_시작일,상품명,카테고리,입고수량,주문수량,판매주문수량,판매수량,주문_판매_갭,순재고변동,브랜드
0,2026-08-31,(대용량) 블랑뮤즈 익스트림 모이스처라이징 바디크림 400ml,바디제품,10,10,0,0,0,10,blance muse
1,2023-04-03,(문빈에디션) 제이준 잇 비건 탠지 워터리 패드 세트,마스크팩,2880,7140,4260,2880,1380,0,JAYJUN
2,2023-07-03,(인터내셔널) 인텐시브 샤이닝 마스크,마스크,600,1390,790,600,190,0,JAYJUN
3,2023-06-12,(인터내셔널) 폴루션 프루프 루미너스 마스크,마스크팩,600,1370,770,600,170,0,JAYJUN
4,2023-07-31,(인터내셔널) 폴루션 프루프 루미너스 마스크,마스크팩,4500,10850,6350,4500,1850,0,JAYJUN


In [10]:
from IPython.display import display
import numpy as np
import pandas as pd

# ==========================================
# [2단계] 전체 주간 그리드 생성, 입력 피처(X) 및 타깃(y) 생성 (display 적용)
# ==========================================

df_wk = weekly_df.copy()
df_wk["주차"] = pd.to_datetime(df_wk["기준주차_시작일"])
df_wk = df_wk.sort_values(["상품명", "주차"]).reset_index(drop=True)

# 1. 전체 상품 × 전체 주차(월요일 시작, W-MON) 완전 그리드 생성
all_weeks = pd.date_range(
    start=df_wk["주차"].min(), end=df_wk["주차"].max(), freq="W-MON"
)
products = df_wk["상품명"].unique()
grid = pd.MultiIndex.from_product(
    [products, all_weeks], names=["상품명", "주차"]
).to_frame(index=False)

# 상품별 메타정보(카테고리, 브랜드) 결합
meta = df_wk.groupby("상품명", as_index=False)[["카테고리", "브랜드"]].first()
df_feat = grid.merge(
    df_wk.drop(
        columns=["카테고리", "브랜드", "기준주차_시작일"], errors="ignore"
    ),
    on=["상품명", "주차"],
    how="left",
).merge(meta, on="상품명", how="left")

# 거래가 없던 빈 주차 수량 컬럼은 0으로 채움
num_fill_zero = [
    "입고수량",
    "주문수량",
    "판매주문수량",
    "판매수량",
    "주문_판매_갭",
    "순재고변동",
]
for col in num_fill_zero:
  df_feat[col] = df_feat[col].fillna(0)

df_feat = df_feat.sort_values(["상품명", "주차"]).reset_index(drop=True)
g = df_feat.groupby("상품명")

# 2. 시차(Lag) 및 이동통계(Rolling) 피처 생성 (미래 누수 방지를 위해 shift(1) 기준)
for lag in [1, 2, 4, 8, 12, 52]:
  df_feat[f"lag_{lag}_주문"] = g["판매주문수량"].shift(lag)

for win in [4, 12, 26]:
  df_feat[f"roll_mean_{win}_주문"] = g["판매주문수량"].transform(
      lambda s: s.shift(1).rolling(win, min_periods=1).mean()
  )
  df_feat[f"roll_std_{win}_주문"] = (
      g["판매주문수량"]
      .transform(lambda s: s.shift(1).rolling(win, min_periods=1).std())
      .fillna(0)
  )

# 주문-판매 갭 피처
df_feat["lag_1_갭"] = g["주문_판매_갭"].shift(1)
df_feat["roll_mean_4_갭"] = g["주문_판매_갭"].transform(
    lambda s: s.shift(1).rolling(4, min_periods=1).mean()
)
df_feat["roll_mean_12_갭"] = g["주문_판매_갭"].transform(
    lambda s: s.shift(1).rolling(12, min_periods=1).mean()
)

# 달력 피처
df_feat["연도"] = df_feat["주차"].dt.year
df_feat["월"] = df_feat["주차"].dt.month
df_feat["주차_번호"] = df_feat["주차"].dt.isocalendar().week.astype(int)

# 3. 미래 누적 타깃(y) 생성 (데이터 끝부분 미관측 미래 구간은 NaN 처리)
max_week = df_feat["주차"].max()

df_feat["y_short_1w"] = g["판매주문수량"].shift(-1)

df_feat["y_short_4w"] = sum(
    g["판매주문수량"].shift(-k).fillna(0) for k in range(1, 5)
)
df_feat.loc[
    df_feat["주차"] > max_week - pd.Timedelta(weeks=4), "y_short_4w"
] = np.nan

df_feat["y_mid_12w"] = sum(
    g["판매주문수량"].shift(-k).fillna(0) for k in range(1, 13)
)
df_feat.loc[
    df_feat["주차"] > max_week - pd.Timedelta(weeks=12), "y_mid_12w"
] = np.nan

df_feat["y_long_52w"] = sum(
    g["판매주문수량"].shift(-k).fillna(0) for k in range(1, 53)
)
df_feat.loc[
    df_feat["주차"] > max_week - pd.Timedelta(weeks=52), "y_long_52w"
] = np.nan

feature_cols = [
    "lag_1_주문",
    "lag_2_주문",
    "lag_4_주문",
    "lag_8_주문",
    "lag_12_주문",
    "lag_52_주문",
    "roll_mean_4_주문",
    "roll_std_4_주문",
    "roll_mean_12_주문",
    "roll_std_12_주문",
    "roll_mean_26_주문",
    "roll_std_26_주문",
    "lag_1_갭",
    "roll_mean_4_갭",
    "roll_mean_12_갭",
    "연도",
    "월",
    "주차_번호",
]

target_cols = ["y_short_1w", "y_short_4w", "y_mid_12w", "y_long_52w"]

step2_summary = pd.DataFrame([{
    "전체_그리드_행수": len(df_feat),
    "전체_그리드_열수": df_feat.shape[1],
    "고유_상품수": len(products),
    "전체_주차수(W-MON)": len(all_weeks),
    "입력피처(X)_개수": len(feature_cols),
    "y_short_1w_유효행": int(df_feat["y_short_1w"].notna().sum()),
    "y_short_4w_유효행": int(df_feat["y_short_4w"].notna().sum()),
    "y_mid_12w_유효행": int(df_feat["y_mid_12w"].notna().sum()),
    "y_long_52w_유효행": int(df_feat["y_long_52w"].notna().sum()),
}])

display("=== [2단계] 피처 및 타깃 생성 요약 ===")
display(step2_summary)

display("=== 생성된 df_feat 일부 컬럼 미리보기(상위 5행) ===")
preview_cols = [
    "상품명",
    "주차",
    "판매주문수량",
    "lag_1_주문",
    "roll_mean_4_주문",
    "lag_1_갭",
    "y_short_1w",
    "y_short_4w",
]
display(df_feat[preview_cols].head(5))

'=== [2단계] 피처 및 타깃 생성 요약 ==='

,전체_그리드_행수,전체_그리드_열수,고유_상품수,전체_주차수(W-MON),입력피처(X)_개수,y_short_1w_유효행,y_short_4w_유효행,y_mid_12w_유효행,y_long_52w_유효행
0,56990,32,278,205,18,56712,55878,53654,42534


'=== 생성된 df_feat 일부 컬럼 미리보기(상위 5행) ==='

,상품명,주차,판매주문수량,lag_1_주문,roll_mean_4_주문,lag_1_갭,y_short_1w,y_short_4w
0,(대용량) 블랑뮤즈 익스트림 모이스처라이징 바디크림 400ml,2023-01-02,0.0,NaN,NaN,NaN,0.0,0.0
1,(대용량) 블랑뮤즈 익스트림 모이스처라이징 바디크림 400ml,2023-01-09,0.0,0.0,0.0,0.0,0.0,0.0
2,(대용량) 블랑뮤즈 익스트림 모이스처라이징 바디크림 400ml,2023-01-16,0.0,0.0,0.0,0.0,0.0,0.0
3,(대용량) 블랑뮤즈 익스트림 모이스처라이징 바디크림 400ml,2023-01-23,0.0,0.0,0.0,0.0,0.0,0.0
4,(대용량) 블랑뮤즈 익스트림 모이스처라이징 바디크림 400ml,2023-01-30,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
from IPython.display import display
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# ==========================================
# [3단계] 창고 제외 v3 데이터 기준 모델 학습 및 기간별(80:20) 오차 검증
# ==========================================

horizons = [
    ("단기_1주", "y_short_1w"),
    ("단기_4주", "y_short_4w"),
    ("중기_12주", "y_mid_12w"),
    ("장기_52주", "y_long_52w"),
]

eval_rows = []
pred_store = {}

for label, target_col in horizons:
  valid_sub = (
      df_feat[df_feat[target_col].notna()]
      .sort_values(["주차", "상품명"])
      .reset_index(drop=True)
  )

  # 시간순 80:20 분할을 위한 컷오프 주차 계산
  c_idx = int(len(valid_sub) * 0.8)
  c_week = valid_sub.loc[c_idx, "주차"]

  tr_sub = valid_sub[valid_sub["주차"] < c_week].copy()
  te_sub = valid_sub[valid_sub["주차"] >= c_week].copy()

  X_tr, y_tr = tr_sub[feature_cols], tr_sub[target_col]
  X_te, y_te = te_sub[feature_cols], te_sub[target_col]

  model = HistGradientBoostingRegressor(
      loss="squared_error",
      max_iter=300,
      learning_rate=0.05,
      max_depth=6,
      min_samples_leaf=20,
      random_state=42,
  )
  model.fit(X_tr, y_tr)

  y_pred_te = np.clip(model.predict(X_te), 0, None)
  pred_store[target_col] = (te_sub, y_te, y_pred_te, model)

  mae = mean_absolute_error(y_te, y_pred_te)
  rmse = root_mean_squared_error(y_te, y_pred_te)
  actual_sum = float(np.sum(y_te))
  abs_err_sum = float(np.sum(np.abs(y_te - y_pred_te)))
  wmape = (abs_err_sum / actual_sum * 100.0) if actual_sum > 0 else np.nan

  eval_rows.append({
      "예측구간": label,
      "타깃컬럼": target_col,
      "컷오프_주차": c_week.strftime("%Y-%m-%d"),
      "Train_행수": len(tr_sub),
      "Test_행수": len(te_sub),
      "MAE": round(mae, 2),
      "RMSE": round(rmse, 2),
      "WMAPE(%)": round(wmape, 2),
  })

eval_df = pd.DataFrame(eval_rows)
display("=== [3단계] 창고 제외(v3) 기간별 80:20 검증 결과 ===")
display(eval_df)

'=== [3단계] 창고 제외(v3) 기간별 80:20 검증 결과 ==='

,예측구간,타깃컬럼,컷오프_주차,Train_행수,Test_행수,MAE,RMSE,WMAPE(%)
0,단기_1주,y_short_1w,2026-02-16,45314,11398,86.81,496.20,486.28
1,단기_4주,y_short_4w,2026-01-26,44480,11398,217.81,1133.40,305.80
2,중기_12주,y_mid_12w,2025-12-15,42812,10842,646.44,2431.54,300.63
3,장기_52주,y_long_52w,2025-05-05,33916,8618,2655.56,8288.29,179.61


In [12]:
from IPython.display import display
import numpy as np
import pandas as pd

# ==========================================
# [4단계] 최신 기준 예측값 확인 및 과거 '주문_판매_갭' 분석 (display 적용)
# ==========================================

latest_week = df_feat["주차"].max()
latest_df = (
    df_feat[df_feat["주차"] == latest_week]
    .sort_values(["카테고리", "상품명"])
    .reset_index(drop=True)
)

latest_out = latest_df[["상품명", "카테고리", "브랜드", "주차", "판매주문수량"]].copy()
latest_out["주차"] = latest_out["주차"].dt.strftime("%Y-%m-%d")

# 3단계에서 학습한 모델(pred_store)로 최신 기준 주차의 미래 구간 예측값 산출
for label, target_col in horizons:
  _, _, _, fitted_model = pred_store[target_col]
  latest_out[f"예측_{label}"] = (
      np.clip(fitted_model.predict(latest_df[feature_cols]), 0, None).round(1)
  )

display(f"=== [4단계] 최신 기준 주차({latest_week.strftime('%Y-%m-%d')}) 예측 상위 15개 품목 ===")
display(latest_out.sort_values("예측_단기_4주", ascending=False).head(15))

# 과거 '주문_판매_갭'이 컸던 상위 15개 품목(창고 제외 v3 기준)
gap_top15 = (
    weekly_df.groupby(["상품명", "카테고리", "브랜드"], as_index=False).agg(
        거래주수=("기준주차_시작일", "count"),
        총_판매주문수량=("판매주문수량", "sum"),
        총_판매수량=("판매수량", "sum"),
        총_주문_판매_갭=("주문_판매_갭", "sum"),
        평균_주문_판매_갭=("주문_판매_갭", "mean"),
    )
    .sort_values("총_주문_판매_갭", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
gap_top15["평균_주문_판매_갭"] = gap_top15["평균_주문_판매_갭"].round(1)

display("=== 과거 누적 '주문_판매_갭' 상위 15개 품목 (창고 제외 v3) ===")
display(gap_top15)

'=== [4단계] 최신 기준 주차(2026-11-30) 예측 상위 15개 품목 ==='

,상품명,카테고리,브랜드,주차,판매주문수량,예측_단기_1주,예측_단기_4주,예측_중기_12주,예측_장기_52주
129,MAXCLINIC 타임리턴 멜라토닌 크림-파우치(1매) 1g,샘플,maxclinic,2026-11-30,0.0,178.1,4348.2,2058.1,0.0
222,MAXCLINIC 골드 캐비어 오일폼(110g),클렌져,maxclinic,2026-11-30,0.0,158.9,1402.8,5550.3,19061.8
127,MAXCLINIC 멜라토닌 샷 스킨 케어 키트,샘플,MAXCLINIC,2026-11-30,0.0,206.7,1391.9,546.8,558.8
57,MAXCLINIC 타임 리턴 마스크 시트 1박스 (28ml×4매),마스크팩,maxclinic,2026-11-30,390.0,178.1,1023.0,1546.5,1857.8
0,BB LAB 유기농 올리브레몬 듀얼샷,건강기능식품,BB LAB,2026-11-30,0.0,178.1,465.6,473.2,1388.9
110,MAXCLINIC 멜라토닌샷 미스트(100ml),미스트,maxclinic,2026-11-30,0.0,158.9,363.1,15.2,6640.8
201,MAXCLINIC 타임 리턴 크림 (50g),크림,maxclinic,2026-11-30,0.0,178.1,328.2,339.2,510.3
143,에프엔리퍼브릭 리쥬셀 글로우 비비 선 크림 (40ml),선케어,FNR,2026-11-30,0.0,178.1,322.8,276.7,1345.5
140,MAXCLINIC 멜라토닌샷 톤업 선스크린(50ml),선케어,maxclinic,2026-11-30,0.0,178.1,303.2,537.8,5260.4
141,MAXCLINIC 멜라토닌샷 워터리 선스크린(50ml),선케어,maxclinic,2026-11-30,0.0,201.9,129.6,285.0,829.2


"=== 과거 누적 '주문_판매_갭' 상위 15개 품목 (창고 제외 v3) ==="

,상품명,카테고리,브랜드,거래주수,총_판매주문수량,총_판매수량,총_주문_판매_갭,평균_주문_판매_갭
0,그린티 아이겔페치,아이케어,JAYJUN,29,182290,164640,17650,608.6
1,케어놀로지 리블루 밸런싱 크림 인 미스트 120ml,미스트,케어놀로지,51,224193,213675,10518,206.2
2,리뉴얼 세리번 나이트 4주분,건강기능식품,SERYBOX,63,201072,192010,9062,143.8
3,스킨핏 마스크,마스크팩,JAYJUN,17,100540,91800,8740,514.1
4,케어놀로지 비파인 더 아크 스팟,크림,케어놀로지,32,176511,168596,7915,247.3
5,케어놀로지 리블루 울트라 리페어 크림 50ml,크림,케어놀로지,30,28774,20909,7865,262.2
6,로젤티 아이겔페치,아이케어,JAYJUN,21,110780,103200,7580,361.0
7,케어놀로지 토닝 라이트업 리페어 선크림 10ml,선케어,케어놀로지,8,52220,46300,5920,740.0
8,블랙 물광 하이드로 슬리핑 팩,마스크팩,JAYJUN,21,41180,35400,5780,275.2
9,인텐시브 브라이트닝 슬리핑 팩,마스크팩,JAYJUN,22,46390,40800,5590,254.1


In [13]:
from IPython.display import display
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# ==========================================
# [5단계] 예측 정확도 개선 실험 (로그변환 / L1손실 / 도메인피처 추가)
# ==========================================

df_imp = df_feat.copy().sort_values(["상품명", "주차"]).reset_index(drop=True)
g_imp = df_imp.groupby("상품명")

# 1. 추가 도메인 피처 생성 (모두 과거 shift(1) 기반, 미래 누수 없음)
df_imp["주문_발생_여부_lag1"] = (
    (df_imp["lag_1_주문"].fillna(0) > 0).astype(int)
)
df_imp["최근4주_주문합"] = df_imp["roll_mean_4_주문"].fillna(0) * 4.0
df_imp["추세_4주_대비_26주"] = np.where(
    df_imp["roll_mean_26_주문"].fillna(0) > 0,
    df_imp["roll_mean_4_주문"].fillna(0) / df_imp["roll_mean_26_주문"].fillna(1),
    0.0,
)
df_imp["갭_비율_lag1"] = np.where(
    df_imp["lag_1_주문"].fillna(0) > 0,
    df_imp["lag_1_갭"].fillna(0) / df_imp["lag_1_주문"].fillna(1),
    0.0,
)

# 최근 실제 주문(판매주문수량 > 0) 발생 이후 경과 주수
has_order = df_imp["판매주문수량"] > 0
last_order_week = df_imp["주차"].where(has_order).groupby(df_imp["상품명"]).ffill().groupby(df_imp["상품명"]).shift(1)
df_imp["경과_주수_lag1"] = (
    (df_imp["주차"] - last_order_week).dt.days / 7.0
).fillna(999).clip(lower=0)

ext_feature_cols = feature_cols + [
    "주문_발생_여부_lag1",
    "최근4주_주문합",
    "추세_4주_대비_26주",
    "갭_비율_lag1",
    "경과_주수_lag1",
]

experiments = [
    ("실험A_Log1p_L2", "squared_error", True, feature_cols),
    ("실험B_AbsoluteError_L1", "absolute_error", False, feature_cols),
    ("실험C_피처확장_Log1p_L2", "squared_error", True, ext_feature_cols),
]

imp_rows = []
for exp_name, loss_fn, use_log1p, f_cols in experiments:
  for label, target_col in horizons:
    valid_sub = (
        df_imp[df_imp[target_col].notna()]
        .sort_values(["주차", "상품명"])
        .reset_index(drop=True)
    )
    c_idx = int(len(valid_sub) * 0.8)
    c_week = valid_sub.loc[c_idx, "주차"]

    tr_sub = valid_sub[valid_sub["주차"] < c_week].copy()
    te_sub = valid_sub[valid_sub["주차"] >= c_week].copy()

    X_tr, X_te = tr_sub[f_cols], te_sub[f_cols]
    y_tr_raw = tr_sub[target_col].values
    y_te = te_sub[target_col].values

    y_tr_fit = np.log1p(np.clip(y_tr_raw, 0, None)) if use_log1p else y_tr_raw

    model = HistGradientBoostingRegressor(
        loss=loss_fn,
        max_iter=300,
        learning_rate=0.05,
        max_depth=6,
        min_samples_leaf=20,
        random_state=42,
    )
    model.fit(X_tr, y_tr_fit)

    raw_pred = model.predict(X_te)
    y_pred_te = np.clip(np.expm1(raw_pred) if use_log1p else raw_pred, 0, None)

    mae = mean_absolute_error(y_te, y_pred_te)
    rmse = root_mean_squared_error(y_te, y_pred_te)
    actual_sum = float(np.sum(y_te))
    abs_err_sum = float(np.sum(np.abs(y_te - y_pred_te)))
    wmape = (abs_err_sum / actual_sum * 100.0) if actual_sum > 0 else np.nan

    imp_rows.append({
        "실험명": exp_name,
        "예측구간": label,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "WMAPE(%)": round(wmape, 2),
    })

imp_df = pd.DataFrame(imp_rows)

# 기존 베이스라인(eval_df)과 WMAPE(%) 피벗 비교
base_pivot = eval_df.set_index("예측구간")[["WMAPE(%)", "MAE"]].rename(
    columns={"WMAPE(%)": "Base_WMAPE(%)", "MAE": "Base_MAE"}
)
exp_pivot = imp_df.pivot(index="예측구간", columns="실험명", values="WMAPE(%)").round(2)
comp_wmape = base_pivot.join(exp_pivot).reindex(["단기_1주", "단기_4주", "중기_12주", "장기_52주"])

display("=== [5단계] 구간별 WMAPE(%) 개선 실험 비교표 ===")
display(comp_wmape)

display("=== 전체 실험 상세 지표(MAE / RMSE / WMAPE) ===")
display(imp_df)

'=== [5단계] 구간별 WMAPE(%) 개선 실험 비교표 ==='

,Base_WMAPE(%),Base_MAE,실험A_Log1p_L2,실험B_AbsoluteError_L1,실험C_피처확장_Log1p_L2
예측구간,,,,,
단기_1주,486.28,86.81,100.57,100.0,100.55
단기_4주,305.80,217.81,100.08,100.0,99.62
중기_12주,300.63,646.44,97.37,97.9,96.96
장기_52주,179.61,2655.56,102.20,100.5,102.84


'=== 전체 실험 상세 지표(MAE / RMSE / WMAPE) ==='

,실험명,예측구간,MAE,RMSE,WMAPE(%)
0,실험A_Log1p_L2,단기_1주,17.95,493.45,100.57
1,실험A_Log1p_L2,단기_4주,71.28,995.42,100.08
2,실험A_Log1p_L2,중기_12주,209.38,2066.77,97.37
3,실험A_Log1p_L2,장기_52주,1511.12,7628.24,102.20
4,실험B_AbsoluteError_L1,단기_1주,17.85,493.49,100.00
5,실험B_AbsoluteError_L1,단기_4주,71.23,998.04,100.00
6,실험B_AbsoluteError_L1,중기_12주,210.51,2056.60,97.90
7,실험B_AbsoluteError_L1,장기_52주,1485.89,7663.46,100.50
8,실험C_피처확장_Log1p_L2,단기_1주,17.95,493.43,100.55
9,실험C_피처확장_Log1p_L2,단기_4주,70.96,992.93,99.62
